In [2]:
import cv2
import mediapipe as mp
import numpy as np

# --- 📌 計算角度的函式 ---
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle
# -----------------------------

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

# 🌟 新增：預設模式為 '1' (大樹式)
current_mode = '1' 

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
            try:
                landmarks = results.pose_landmarks.landmark
                
                # 抓取座標
                hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
                shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
                elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
                wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
                knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
                ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

                # 計算角度
                elbow_angle = calculate_angle(shoulder, elbow, wrist)
                shoulder_angle = calculate_angle(hip, shoulder, elbow)
                knee_angle = calculate_angle(hip, knee, ankle)
                leg_angle = calculate_angle(shoulder, hip, knee)
                
                # 🌟 根據目前的模式執行對應邏輯
                match current_mode:
                    case '1':
                        cv2.putText(image, 'Mode: Tree Pose', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2, cv2.LINE_AA)
                        if elbow_angle < 160:
                            cv2.putText(image, 'ERROR: Straighten your arm!', (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif shoulder_angle < 75:
                            cv2.putText(image, 'ERROR: Rise', (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif shoulder_angle > 105:
                            cv2.putText(image, 'ERROR: Put Down', (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            cv2.putText(image, 'ARM PERFECT', (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

                        if leg_angle > 110:
                            cv2.putText(image, 'ERROR: Raise Your Leg!', (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif leg_angle < 75:
                            cv2.putText(image, 'ERROR: Down Your Leg', (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        elif knee_angle < 160:
                            cv2.putText(image, 'ERROR: Straighten Your leg', (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            cv2.putText(image, 'LEG PERFECT', (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
                            
                    case '2': 
                        # 這是深蹲模式，你可以換成深蹲的判斷邏輯
                        cv2.putText(image, 'Mode: Squat', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 100, 0), 2, cv2.LINE_AA)
                        if knee_angle > 120:
                            cv2.putText(image, 'ERROR: Squat low', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                        else:
                            cv2.putText(image, 'PerFECT', (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

            except Exception as e: 
                pass

        # 🌟 在畫面上提示使用者可以按哪些鍵
        cv2.putText(image, 'Press "1" for Tree Pose | "2" for Squat | "q" to Quit', (10, 450), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)

        cv2.imshow('Yoga Test', image)
        
        # 🌟 擷取鍵盤輸入來即時切換模式
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'):    # 按 q 離開
            break
        elif key == ord('1'):  # 按 1 切換到大樹式
            current_mode = '1'
        elif key == ord('2'):  # 按 2 切換到深蹲
            current_mode = '2'

cap.release()
cv2.destroyAllWindows()